# 56 - Speckle Confidence Adaptive R Gate

This notebook documents and smoke-tests the optional ultrasound confidence layer for the strict Python UltraTimTrack pipeline.

The working UltraTimTrack path remains fixed-R by default. Adaptive measurement covariance is enabled only when `--adaptive-r` is used.

## Repository modules

- Kalman: `ultrasound_tracker.ultratimtrack_matlab_2state`
- KLT / optical flow: `ultrasound_tracker.ultratrack_klt`
- Hough / Frangi / TimTrack detection: `ultrasound_tracker.timtrack_hough`, `ultrasound_tracker.matlab_timtrack`, `ultrasound_tracker.frangi_detector`
- Geometry: `ultrasound_tracker.geometry`, `ultrasound_tracker.final_output`
- Video runners: `scripts/run_strict_ultratimtrack_video.py`, `scripts/run_ultratimtrack_adaptive_confidence.py`
- Confidence layer: `ultrasound_tracker.speckle_confidence`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from ultrasound_tracker.speckle_confidence import (
    SpeckleConfidenceConfig,
    adapt_measurement_covariance,
    combine_confidence_metrics,
    compute_motion_consistency,
    compute_speckle_coherence,
)

cfg = SpeckleConfidenceConfig(block_size=11, stride=16, search_radius=4, min_texture_variance=1.0)
cfg

SpeckleConfidenceConfig(block_size=11, stride=16, search_radius=4, min_texture_variance=1.0, zncc_low=0.45, zncc_high=0.9, confidence_floor=0.05, confidence_ceiling=1.0, r_min_scale=0.5, r_max_scale=20.0, r_gamma=1.5, use_zncc=True, min_points=3, motion_spread_scale_px=3.0, forward_backward_scale_px=2.0, min_feature_peaks_for_full_conf=5, feature_peak_scale=25.0, min_mask_density=0.002, max_mask_density=0.35, plausible_alpha_range_deg=(5.0, 85.0), plausible_pennation_range_deg=(0.0, 45.0), plausible_length_range_px=(20.0, 2000.0), angle_jump_scale_deg=8.0, length_jump_scale_px=60.0, weights={'speckle': 0.35, 'motion': 0.25, 'feature': 0.25, 'geometry': 0.15})

## Synthetic confidence sanity checks

Identical textured ultrasound-like patches should have high speckle confidence. Decorrelation or incoherent local motion should reduce confidence. Low confidence means the Kalman filter should trust the measurement less, not that the frame must be discarded.

In [2]:
rng = np.random.default_rng(42)
image = rng.integers(0, 255, size=(96, 96), dtype=np.uint8)
decorrelated = rng.integers(0, 255, size=(96, 96), dtype=np.uint8)

same = compute_speckle_coherence(image, image.copy(), roi=(0, 0, 96, 96), config=cfg)
bad = compute_speckle_coherence(image, decorrelated, roi=(0, 0, 96, 96), config=cfg)

points = rng.uniform(0, 100, size=(60, 2)).astype(np.float32)
coherent_motion = compute_motion_consistency(points, points + np.array([2.0, -1.0], dtype=np.float32), config=cfg)
random_motion = compute_motion_consistency(points, points + rng.normal(0, 10, size=(60, 2)), config=cfg)

summary = {
    'same_speckle_confidence': same['speckle_confidence'],
    'decorrelated_speckle_confidence': bad['speckle_confidence'],
    'coherent_motion': coherent_motion['motion_consistency'],
    'random_motion': random_motion['motion_consistency'],
}
summary

{'same_speckle_confidence': 1.0,
 'decorrelated_speckle_confidence': 0.05,
 'coherent_motion': 1.0,
 'random_motion': 0.11245280049607696}

In [3]:
metrics = {
    'speckle_confidence': same['speckle_confidence'],
    'motion_consistency': coherent_motion['motion_consistency'],
    'feature_reliability': 0.9,
    'geometry_stability': 0.95,
}
confidence = combine_confidence_metrics(metrics, config=cfg)
R_base = np.diag([100.0, 3.05529211])
R_t = adapt_measurement_covariance(R_base, confidence, cfg)
confidence, np.diag(R_t)

(0.9665385228106151, array([61.9358204 ,  1.89232023]))

## Optional video gate

Set `RUN_VIDEO_GATE = True` to run a short adaptive-R video smoke test. For a full sequence, remove `--limit 30`.

In [4]:
import subprocess

RUN_VIDEO_GATE = False
cmd = [
    str(PROJECT_ROOT / '.venv' / 'bin' / 'python'),
    str(PROJECT_ROOT / 'scripts' / 'run_ultratimtrack_adaptive_confidence.py'),
    str(PROJECT_ROOT / 'data' / 'raw' / 'UltraTimTrack_test.mp4'),
    '--roi-path', str(PROJECT_ROOT / 'data' / 'rois' / 'UltraTimTrack_test_rois.json'),
    '--limit', '30',
    '--seed-frames', '5',
    '--adaptive-r',
    '--save-confidence-plots',
    '--no-annotated-video',
    '--save-overlays', '1',
    '--results-dir', str(PROJECT_ROOT / 'results' / 'adaptive_confidence_notebook'),
]

if RUN_VIDEO_GATE:
    subprocess.run(cmd, check=True)
else:
    print('Set RUN_VIDEO_GATE=True to execute:')
    print(' '.join(cmd))

Set RUN_VIDEO_GATE=True to execute:
/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/run_ultratimtrack_adaptive_confidence.py /Users/grosbedou/PycharmProjects/NDORMS/data/raw/UltraTimTrack_test.mp4 --roi-path /Users/grosbedou/PycharmProjects/NDORMS/data/rois/UltraTimTrack_test_rois.json --limit 30 --seed-frames 5 --adaptive-r --save-confidence-plots --no-annotated-video --save-overlays 1 --results-dir /Users/grosbedou/PycharmProjects/NDORMS/results/adaptive_confidence_notebook


## Interpretation

Adaptive-R mode saves per-frame `speckle_zncc`, `speckle_confidence`, `motion_consistency`, `feature_reliability`, `geometry_stability`, `combined_confidence`, `r_scale`, and `R_t` diagonal values. When confidence is low, `R_t` increases and the Kalman update trusts the measurement less. Fixed-R behavior remains the default.